# The whole pipeline, one frame at a time

This notebook walks one validation frame through every stage of the pipeline, on CPU, with the trained
U-Net of `filament-cnn-baseline` (attached as a notebook source). The full runs live in their own notebooks:

| stage | script (notebook) | what it produces |
|---|---|---|
| train | `solution/cnn_baseline/main.py` (`filament-cnn-baseline`, T4, ~40 min) | `unet.pt` |
| 5-fold OOF | `solution/cnn_baseline/cv.py` (`filament-cnn-baseline-cv`) | OOF maps for all 707 frames |
| test maps | `solution/cnn_baseline/predict.py` (`filament-cnn-baseline-predict`) | single-model test maps |
| post-processing, tuned on OOF | `solution/cnn_baseline/final_join.py` (CPU) | `submission.csv` |
| final model | Swin U-Net with boundary loss (teammate; see the report, section 4.2) | team's final submission |

Every script becomes a Kaggle notebook with `uv run filament run <script>` (tests, build, push, wait, fetch
outputs, format check). Stages below: **disk and limb correction → network → hysteresis instances →
joining by filament physics → component features → score and RLE.**

> Generated by `submission_pipeline/notebook.py` from `solution/cnn_baseline/pipeline.py` and the modules it imports. Edit those, not this notebook: the `%%writefile` cells are copies.

## Modules

The code this notebook imports, written to disk first.

In [ ]:
import os

os.makedirs("cnn_baseline", exist_ok=True)
os.makedirs("dataset", exist_ok=True)
os.makedirs("error_analysis", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("preprocess", exist_ok=True)
os.makedirs("scoring", exist_ok=True)

In [ ]:
%%writefile dataset/__init__.py
"""The competition's data as distributed: where it lives and how the annotations are stored."""

IMAGE_SIZE = 2048  # every GONG H-alpha frame in the competition is IMAGE_SIZE x IMAGE_SIZE
N_TEST_IMAGES = 180

In [ ]:
%%writefile dataset/annotations.py
"""MAGFiLO COCO-style annotations.

One physical image (`file_name`) can carry up to three independent annotation sets from different
annotators. Each set is its own entry in `images`, with an id like `050102-20111116063134Lh`, and the
hosts score predictions against every set separately. Anything that splits data must therefore group
by `file_name`, never by entry id.
"""

import json
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

from pycocotools import mask as mask_util

from dataset import IMAGE_SIZE


@dataclass
class Annotations:
    file_to_entries: dict[str, list[str]]  # file_name -> annotator-image ids
    entry_to_rles: dict[str, list[dict]]  # annotator-image id -> one pycocotools RLE per filament
    n_annotations: int
    n_multi_polygon: int  # filaments drawn as more than one polygon (0 in the 2026 training set)

    def rles_for_file(self, file_name: str) -> list[list[dict]]:
        """Ground truth of one physical image: a list of filament RLEs per annotator."""
        return [self.entry_to_rles.get(entry, []) for entry in self.file_to_entries.get(file_name, [])]


def load_annotations(json_path: str | Path) -> Annotations:
    with open(json_path) as f:
        coco = json.load(f)
    file_to_entries = defaultdict(list)
    for image in coco["images"]:
        file_to_entries[image["file_name"]].append(image["id"])
    entry_to_rles = defaultdict(list)
    n_multi_polygon = 0
    for ann in coco["annotations"]:
        polygons = [p for p in ann["segmentation"] if len(p) >= 6]  # a polygon needs three points
        if not polygons:
            continue
        n_multi_polygon += len(polygons) > 1
        rle = mask_util.merge(mask_util.frPyObjects(polygons, IMAGE_SIZE, IMAGE_SIZE))
        entry_to_rles[ann["image_id"]].append(rle)
    return Annotations(dict(file_to_entries), dict(entry_to_rles), len(coco["annotations"]), n_multi_polygon)

In [ ]:
%%writefile dataset/paths.py
"""Where the competition data, attached notebook outputs and the writable output directory live."""

import os
from pathlib import Path

DATASET_DIR_NAME = "MAGFiLO_1.0_Kaggle_2026"


def input_root() -> Path:
    """Kaggle mounts the competition data and any attached notebook outputs under /kaggle/input.

    FILAMENT_INPUT overrides it, so a notebook can be smoke-tested locally on a synthetic copy.
    """
    return Path(os.environ.get("FILAMENT_INPUT", "/kaggle/input"))


def find_data_root() -> Path:
    """Locate the dataset folder; Kaggle's mount point has changed between releases, so search for it.

    FILAMENT_DATA points straight at the folder and skips the search.
    """
    if os.environ.get("FILAMENT_DATA"):
        return Path(os.environ["FILAMENT_DATA"])
    hits = sorted(input_root().glob(f"**/{DATASET_DIR_NAME}"))
    if not hits:
        raise FileNotFoundError(f"{DATASET_DIR_NAME} not found under {input_root()}")
    return hits[0]


def find_input(pattern: str) -> Path:
    """First file under the input root matching a glob, e.g. a `unet.pt` saved by another notebook."""
    hits = sorted(input_root().glob(f"**/{pattern}"))
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {input_root()}")
    return hits[0]


def output_dir() -> Path:
    """Kaggle's writable directory, else FILAMENT_OUTPUT, else the current directory."""
    kaggle_working = Path("/kaggle/working")
    if kaggle_working.is_dir():
        return kaggle_working
    return Path(os.environ.get("FILAMENT_OUTPUT", "."))

In [ ]:
%%writefile error_analysis/__init__.py
"""Where a submission loses Panoptic Quality, instance by instance.

PQ = sum of IoU over matches / (TP + 0.5 FP + 0.5 FN), a match being IoU > 0.5. Only three things
can go wrong, so the analysis classifies every ground-truth filament and every prediction by how it
relates to the other side (`instances`), aggregates the tables into what-if bounds and breakdowns by
size, contrast, position, site and annotator agreement (`report`), draws the worst cases (`gallery`),
and measures how far apart the human annotators are from each other (`annotators`), which is the
ceiling any model is scored against.

The tables are plain CSVs, so they come back from Kaggle and can be sliced locally without the images.

The notebooks live here too, next to the modules they use, like an approach directory under solution/:
`annotator_agreement.py` and `cnn_errors.py` run with `uv run filament run error_analysis/<name>.py --figures`
and become `filament-error-analysis-<name>`; `python -m error_analysis <outputs dir>` reprints a report locally.
"""

In [ ]:
%%writefile error_analysis/instances.py
"""Per-instance tables: every ground-truth filament and every prediction of a frame, classified.

Ground truth and predictions are linked wherever they share a pixel. The shape of those links, plus
the IoU > 0.5 rule of the metric, gives each instance one status:

    ground truth                      prediction
    matched     IoU > 0.5             matched        IoU > 0.5 with some filament
    near_miss   one overlap, IoU <= 0.5, and that prediction touches nothing else
    fragmented  covered by >= 2 preds  fragment       one of several covering a filament
    merged      swallowed by a pred    merge          covers >= 2 filaments
                that covers others
    missed      no overlap             hallucination  no overlap

Scoring is per annotator-image, so one prediction yields one row per annotator of its frame.
"""

from dataclasses import dataclass

import numpy as np
import pandas as pd
from pycocotools import mask as mask_util

from preprocess.disk import REL_SCALE
from scoring.metric import IOU_THRESHOLD, InstanceCrop, instance_crops


@dataclass
class FrameContext:
    """What the features of a frame's instances are computed from."""

    name: str  # file name, e.g. 20140609195854Bh.jpeg
    flat: np.ndarray  # uint8 limb-darkening-corrected frame (preprocess.disk.flatten)
    centre: tuple[float, float]  # disk centre in pixels
    radius: float  # disk radius in pixels
    prob: np.ndarray | None = None  # uint8 probability * 255 predicted by the model, if any

    @property
    def site(self) -> str:
        return self.name[14:16]

    @property
    def year(self) -> int:
        return int(self.name[:4])


def features(crop: np.ndarray, x: int, y: int, ctx: FrameContext) -> dict:
    """Shape and photometric features of one instance given its bounding-box crop (bool) at (x, y)."""
    ys, xs = np.nonzero(crop)
    h, w = crop.shape
    cx, cy = float(xs.mean()) + x, float(ys.mean()) + y
    if len(xs) > 2:
        eigenvalues = np.linalg.eigvalsh(np.cov(np.stack([xs, ys]).astype(float)))
        elongation = float(np.sqrt(max(eigenvalues[1], 1e-9) / max(eigenvalues[0], 1e-9)))
    else:
        elongation = 1.0
    flat = ctx.flat[y : y + h, x : x + w][crop]
    out = {
        "area": len(xs),
        "cx": cx,
        "cy": cy,
        "elongation": elongation,  # ratio of the principal axes; 1 = round, 10 = thin and long
        "darkness": float(1 - flat.mean() / REL_SCALE),  # 0.12 = 12 % darker than the local background
        "limb": float(np.hypot(cx - ctx.centre[0], cy - ctx.centre[1]) / ctx.radius) if ctx.radius else np.nan,
    }
    if ctx.prob is not None:
        prob = ctx.prob[y : y + h, x : x + w][crop]
        out.update(mean_prob=float(prob.mean() / 255), max_prob=float(prob.max() / 255))
    return out


def overlaps(gt_crops: list[InstanceCrop], labels: np.ndarray, pred_areas: np.ndarray):
    """(intersection, iou) matrices of shape (n_gt, n_pred), from the label map of the predictions."""
    n_pred = len(pred_areas)
    intersection = np.zeros((len(gt_crops), n_pred), dtype=np.int64)
    for i, (x, y, crop) in enumerate(gt_crops):
        h, w = crop.shape
        intersection[i] = np.bincount(labels[y : y + h, x : x + w][crop], minlength=n_pred + 1)[1:]
    gt_areas = np.array([int(crop.sum()) for _, _, crop in gt_crops], dtype=np.int64)
    union = gt_areas[:, None] + np.asarray(pred_areas, dtype=np.int64)[None, :] - intersection
    iou = np.where(union > 0, intersection / np.maximum(union, 1), 0.0)
    return intersection, iou


def classify(iou: np.ndarray) -> tuple[list[str], list[str]]:
    """Status of each ground-truth row and each prediction column of an IoU matrix."""
    touching = iou > 0
    gt_links, pred_links = touching.sum(axis=1), touching.sum(axis=0)
    hit = iou > IOU_THRESHOLD  # at most one per row and per column: two IoUs > 0.5 cannot share a side
    gt_status = []
    for i in range(iou.shape[0]):
        if gt_links[i] == 0:
            gt_status.append("missed")
        elif hit[i].any():
            gt_status.append("matched")
        elif gt_links[i] >= 2:
            gt_status.append("fragmented")
        else:
            gt_status.append("merged" if pred_links[int(iou[i].argmax())] >= 2 else "near_miss")
    pred_status = []
    for j in range(iou.shape[1]):
        if pred_links[j] == 0:
            pred_status.append("hallucination")
        elif hit[:, j].any():
            pred_status.append("matched")
        elif pred_links[j] >= 2:
            pred_status.append("merge")
        else:
            pred_status.append("fragment" if gt_links[int(iou[:, j].argmax())] >= 2 else "near_miss")
    return gt_status, pred_status


def agreement(rles_by_annotator: list[list[dict]]) -> list[np.ndarray]:
    """For each annotator's filaments: how many OTHER annotators drew a matching one (IoU > 0.5)."""
    counts = []
    for a, rles_a in enumerate(rles_by_annotator):
        count = np.zeros(len(rles_a), dtype=int)
        for b, rles_b in enumerate(rles_by_annotator):
            if a != b and rles_a and rles_b:
                iou = mask_util.iou(rles_a, rles_b, [0] * len(rles_b))  # (n_a, n_b)
                count += iou.max(axis=1) > IOU_THRESHOLD
        counts.append(count)
    return counts


def analyze_frame(
    ctx: FrameContext,
    annotators: list[tuple[str, list[dict]]],
    labels: np.ndarray,
    pred_areas: np.ndarray,
    pred_boxes: np.ndarray,
) -> tuple[list[dict], list[dict]]:
    """Rows for every (annotator, filament) and every (annotator, prediction) of one frame.

    `annotators` pairs each annotator-image id with its filament RLEs; `labels`, `pred_areas` and
    `pred_boxes` are the predictions as `postprocess.instances` returns them.
    """
    n_annotators = len(annotators)
    agreements = agreement([rles for _, rles in annotators])
    pred_features = []
    for k, (x, y, w, h) in enumerate(pred_boxes, start=1):
        crop = labels[y : y + h, x : x + w] == k
        pred_features.append({"pred_id": k, **features(crop, int(x), int(y), ctx)})
    frame_info = {"frame": ctx.name, "site": ctx.site, "year": ctx.year, "n_annotators": n_annotators}

    gt_rows, pred_rows = [], []
    for a, (annotator, rles) in enumerate(annotators):
        crops = instance_crops(rles)
        intersection, iou = overlaps(crops, labels, pred_areas)
        gt_status, pred_status = classify(iou)
        for i, (x, y, crop) in enumerate(crops):
            best = int(iou[i].argmax()) if iou.shape[1] else -1
            gt_rows.append(
                {
                    **frame_info,
                    "annotator": annotator,
                    "gt_id": i,
                    **features(crop, x, y, ctx),
                    "agreement": int(agreements[a][i]),  # other annotators who drew this filament too
                    "status": gt_status[i],
                    "n_overlap": int((iou[i] > 0).sum()),
                    "best_iou": float(iou[i, best]) if best >= 0 else 0.0,
                    "best_pred": best + 1 if best >= 0 and iou[i, best] > 0 else 0,
                    "coverage": float(intersection[i].sum() / max(crop.sum(), 1)),  # of its pixels under any pred
                }
            )
        for j, feats in enumerate(pred_features):
            best = int(iou[:, j].argmax()) if iou.shape[0] else -1
            pred_rows.append(
                {
                    **frame_info,
                    "annotator": annotator,
                    **feats,
                    "status": pred_status[j],
                    "n_overlap": int((iou[:, j] > 0).sum()),
                    "best_iou": float(iou[best, j]) if best >= 0 else 0.0,
                    "best_gt": best if best >= 0 and iou[best, j] > 0 else -1,
                    "precision": float(intersection[:, j].sum() / max(feats["area"], 1)),  # of its pixels on any GT
                }
            )
    return gt_rows, pred_rows


FRAME_COLUMNS = ["frame", "site", "year", "n_annotators", "annotator"]
FEATURE_COLUMNS = ["area", "cx", "cy", "elongation", "darkness", "limb", "mean_prob", "max_prob"]
GT_COLUMNS = [
    *FRAME_COLUMNS,
    "gt_id",
    *FEATURE_COLUMNS,
    "agreement",
    "status",
    "n_overlap",
    "best_iou",
    "best_pred",
    "coverage",
]
PRED_COLUMNS = [*FRAME_COLUMNS, "pred_id", *FEATURE_COLUMNS, "status", "n_overlap", "best_iou", "best_gt", "precision"]


def tables(gt_rows: list[dict], pred_rows: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """DataFrames with a fixed column set, so an empty side (no predictions at all) still has a `status`."""
    gt = pd.DataFrame(gt_rows) if gt_rows else pd.DataFrame(columns=GT_COLUMNS)
    pred = pd.DataFrame(pred_rows) if pred_rows else pd.DataFrame(columns=PRED_COLUMNS)
    return gt, pred

In [ ]:
%%writefile models/__init__.py
"""Model architectures, independent of any training recipe. Each approach under solution/ picks one."""

In [ ]:
%%writefile models/unet.py
"""A plain U-Net: the ordinary convolutional encoder-decoder for segmentation, written from scratch.

No pretrained backbone, no attention. Each level is two 3x3 conv-BN-ReLU; the encoder halves the
resolution with max-pooling, the decoder doubles it bilinearly and concatenates the skip connection.
It is fully convolutional, so it can train on tiles and predict whole 2048 x 2048 frames.
"""

import torch
from torch import nn
from torch.nn import functional as F


def conv_block(in_channels: int, out_channels: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """`depth` down-sampling steps; channel widths base, 2*base, ... at each level. Output: logits."""

    def __init__(self, in_channels: int = 2, out_channels: int = 1, base: int = 32, depth: int = 4):
        super().__init__()
        widths = [base * 2**level for level in range(depth + 1)]
        self.encoders = nn.ModuleList()
        previous = in_channels
        for width in widths:
            self.encoders.append(conv_block(previous, width))
            previous = width
        self.decoders = nn.ModuleList(
            conv_block(widths[level + 1] + widths[level], widths[level]) for level in reversed(range(depth))
        )
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Conv2d(base, out_channels, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skips = []
        for level, encoder in enumerate(self.encoders):
            x = encoder(x if level == 0 else self.pool(x))
            skips.append(x)
        x = skips.pop()
        for decoder in self.decoders:
            skip = skips.pop()
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = decoder(torch.cat([x, skip], dim=1))
        return self.head(x)

In [ ]:
%%writefile preprocess/__init__.py
"""Image preprocessing shared by the approaches: reading frames, solar-disk geometry, limb darkening."""

In [ ]:
%%writefile preprocess/disk.py
"""Solar-disk geometry and limb-darkening removal for full-disk H-alpha frames."""

import cv2
import numpy as np

REL_SCALE = 200  # `flatten` stores image / background as uint8, with the background level at REL_SCALE
OFF_DISK = 255  # value `flatten` assigns outside the (shrunk) disk


def disk_geometry(image: np.ndarray) -> tuple[float, float, float]:
    """(centre x, centre y, radius) of the solar disk in pixels; radius 0 when no disk is found.

    The sky around the disk is grey rather than black in many frames, hence Otsu instead of a fixed
    threshold. Centre and radius come from the blob's centroid and area, which ignores prominences.
    """
    blurred = cv2.GaussianBlur(image, (0, 0), 5)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, _, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if n < 2:
        return image.shape[1] / 2, image.shape[0] / 2, 0.0
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return float(centroids[k][0]), float(centroids[k][1]), float(np.sqrt(stats[k, cv2.CC_STAT_AREA] / np.pi))


def disk_mask(image: np.ndarray, shrink: float = 1.0) -> np.ndarray:
    """uint8 {0, 1} mask of the solar disk, its radius scaled by `shrink`."""
    cx, cy, radius = disk_geometry(image)
    mask = np.zeros_like(image, dtype=np.uint8)
    if radius > 0:
        cv2.circle(mask, (round(cx), round(cy)), int(radius * shrink), 1, -1)
    return mask


def flatten(image: np.ndarray, shrink: float = 0.96, background_sigma: float = 40, smooth_sigma: float = 2):
    """Remove limb darkening: uint8 map of REL_SCALE * image / smooth background, OFF_DISK off-disk.

    The background is smooth, so it is estimated at 1/8 resolution. It is a normalised convolution
    (blurred image / blurred disk mask), which keeps the dark sky from bleeding into the limb.
    `shrink` drops the outermost ring, where the limb itself would read as a dark structure.
    """
    size = image.shape[0]
    disk = disk_mask(image, shrink).astype(np.float32)
    image_f = image.astype(np.float32)  # uint8 arithmetic would wrap around
    small = (size // 8, size // 8)
    weight = cv2.GaussianBlur(cv2.resize(disk, small, interpolation=cv2.INTER_AREA), (0, 0), background_sigma / 8)
    blurred = cv2.GaussianBlur(
        cv2.resize(image_f * disk, small, interpolation=cv2.INTER_AREA), (0, 0), background_sigma / 8
    )
    background = cv2.resize(blurred / np.maximum(weight, 1e-3), (size, size), interpolation=cv2.INTER_LINEAR)
    # filaments are ~10-20 px wide, so a light blur only tames the chromospheric texture
    smooth = cv2.GaussianBlur(image_f, (0, 0), smooth_sigma) if smooth_sigma > 0 else image_f
    rel = np.clip(REL_SCALE * smooth / np.maximum(background, 1.0), 0, 255).astype(np.uint8)
    rel[disk == 0] = OFF_DISK
    return rel

In [ ]:
%%writefile preprocess/images.py
"""Reading the competition's JPEG frames."""

from pathlib import Path

import cv2
import numpy as np


def read_gray(path: str | Path) -> np.ndarray:
    """uint8 (H, W). The frames are grayscale; reading them as colour would only triplicate the data."""
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise OSError(f"cannot read {path}")
    return image

In [ ]:
%%writefile scoring/__init__.py
"""How a submission is scored: the mask encoding and CSV the hosts expect, their Panoptic Quality
metric, and a format check to run before spending one of the five daily submissions."""

In [ ]:
%%writefile scoring/metric.py
"""Panoptic Quality as defined by the hosts' self-evaluation notebook.

    PQ = sum of IoU over true-positive pairs / (|TP| + 0.5 |FP| + 0.5 |FN|)

A ground-truth/prediction pair is a true positive when IoU > 0.5; unmatched predictions are false
positives and unmatched ground-truth filaments are false negatives. Scores are accumulated over
annotator-images, i.e. the same prediction is compared with every annotator's set for that image.

Predictions are passed as a label map, so they cannot overlap and a filament matches at most one
prediction. Overlaps are counted inside each ground-truth bounding box only, which makes evaluating
a 2048 x 2048 frame a matter of milliseconds.
"""

import numpy as np
from pycocotools import mask as mask_util

IOU_THRESHOLD = 0.5

InstanceCrop = tuple[int, int, np.ndarray]  # x, y, boolean mask of the bounding box


def instance_crops(rles: list[dict]) -> list[InstanceCrop]:
    """Ground-truth RLEs -> bounding-box crops, the form `PanopticQuality.update` consumes."""
    crops = []
    for rle in rles:
        x, y, w, h = (int(v) for v in mask_util.toBbox(rle))
        if w > 0 and h > 0:
            crops.append((x, y, mask_util.decode(rle)[y : y + h, x : x + w].astype(bool)))
    return crops


class PanopticQuality:
    def __init__(self):
        self.iou_sum, self.tp, self.fp, self.fn = 0.0, 0, 0, 0

    def update(self, gt_crops: list[InstanceCrop], labels: np.ndarray, pred_areas: np.ndarray) -> None:
        """Add one annotator-image. `labels` holds 0 for background and 1..K for the K predictions."""
        n_pred = len(pred_areas)
        if not gt_crops or n_pred == 0:
            self.fp += n_pred
            self.fn += len(gt_crops)
            return
        pred_hit = np.zeros(n_pred, dtype=bool)
        for x, y, crop in gt_crops:
            h, w = crop.shape
            intersection = np.bincount(labels[y : y + h, x : x + w][crop], minlength=n_pred + 1)[1:]
            iou = intersection / (crop.sum() + pred_areas - intersection)
            best = int(iou.argmax())
            if iou[best] > IOU_THRESHOLD:
                self.tp += 1
                self.iou_sum += float(iou[best])
                pred_hit[best] = True
            else:
                self.fn += 1
        self.fp += int((~pred_hit).sum())

    @property
    def value(self) -> float:
        denominator = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.iou_sum / denominator if denominator > 0 else 0.0

    def summary(self) -> dict:
        """PQ with its factors: SQ (mean IoU of matches) and RQ (an F1 over instances)."""
        sq = self.iou_sum / self.tp if self.tp else 0.0
        return {
            "pq": round(self.value, 4),
            "sq": round(sq, 4),
            "rq": round(self.value / sq, 4) if sq else 0.0,
            "tp": self.tp,
            "fp": self.fp,
            "fn": self.fn,
        }

In [ ]:
%%writefile scoring/rle.py
"""COCO run-length encoding helpers. The submission stores only the compressed `counts` string."""

import numpy as np
from pycocotools import mask as mask_util

from dataset import IMAGE_SIZE


def encode(mask: np.ndarray) -> str:
    """Binary (H, W) mask -> compressed RLE counts."""
    return mask_util.encode(np.asfortranarray(mask.astype(np.uint8)))["counts"].decode("ascii")


def decode(counts: str, size: int = IMAGE_SIZE) -> np.ndarray:
    """Compressed RLE counts -> uint8 (size, size) mask."""
    return mask_util.decode({"size": [size, size], "counts": counts.encode("ascii")})


def encode_component(labels: np.ndarray, label: int, box: tuple[int, int, int, int]) -> str:
    """Encode one instance of a label map, touching only its bounding box (x, y, w, h)."""
    x, y, w, h = box
    mask = np.zeros(labels.shape, dtype=np.uint8, order="F")
    mask[y : y + h, x : x + w] = labels[y : y + h, x : x + w] == label
    return mask_util.encode(mask)["counts"].decode("ascii")


def is_valid(counts: str, size: int = IMAGE_SIZE) -> bool:
    """True if `counts` is well-formed and its runs cover the image exactly.

    pycocotools decodes garbage without complaint, so the string is parsed here: each number is a
    little-endian sequence of 5-bit groups (bit 5 = continuation, bit 4 of the last group = sign),
    and from the fourth number on values are deltas against the number two positions earlier.
    """
    runs, pos = [], 0
    try:
        while pos < len(counts):
            value, group, more = 0, 0, True
            while more:
                char = ord(counts[pos]) - 48
                if not 0 <= char < 64:
                    return False
                value |= (char & 0x1F) << (5 * group)
                more = bool(char & 0x20)
                pos += 1
                group += 1
                if not more and (char & 0x10):
                    value |= -1 << (5 * group)
            if len(runs) > 2:
                value += runs[-2]
            if value < 0:
                return False
            runs.append(value)
    except IndexError:  # the string ended in the middle of a number
        return False
    return sum(runs) == size * size

In [ ]:
%%writefile scoring/submission.py
"""Submission file: one row per predicted filament, `filament_id` = `<image id>_<n>`."""

from pathlib import Path

import numpy as np
import pandas as pd

from scoring import rle

COLUMNS = ["filament_id", "segmentation_rle"]


def rows_from_label_map(image_id: str, labels: np.ndarray, boxes: np.ndarray) -> list[dict]:
    """Rows for the instances 1..K of a label map; `boxes[k - 1]` is instance k as (x, y, w, h)."""
    return [
        {"filament_id": f"{image_id}_{k}", "segmentation_rle": rle.encode_component(labels, k, tuple(box))}
        for k, box in enumerate(boxes, start=1)
    ]


def write(rows: list[dict], path: str | Path) -> pd.DataFrame:
    frame = pd.DataFrame(rows, columns=COLUMNS)
    frame.to_csv(path, index=False)
    return frame

In [ ]:
%%writefile cnn_baseline/__init__.py
"""CNN baseline: a plain U-Net trained from scratch on tiles, predicting a filament probability map;
instances are its connected components. The simplest learned approach, to see how far it gets."""

In [ ]:
%%writefile cnn_baseline/data.py
"""Frames as network input, soft filament targets, and tile sampling for training."""

import random
from pathlib import Path

import cv2
import numpy as np
import torch
from pycocotools import mask as mask_util
from torch.utils.data import Dataset

from dataset.annotations import Annotations
from preprocess.disk import OFF_DISK, flatten
from preprocess.images import read_gray

OFF_DISK_VALUE = OFF_DISK / 255.0  # what the flattened channel holds outside the disk


def split_files(files: list[str], validation_fraction: float, seed: int) -> tuple[list[str], list[str]]:
    """(training, validation) file names: a deterministic shuffle of the physical frames.

    Splitting by frame keeps the up-to-three annotation sets of one frame on the same side.
    """
    files = sorted(files)
    random.Random(seed).shuffle(files)
    n_val = max(1, round(len(files) * validation_fraction))
    return files[n_val:], files[:n_val]


def fold_files(files: list[str], n_folds: int, seed: int) -> list[list[str]]:
    """`n_folds` disjoint groups of physical frames covering all of `files` (the shuffle of `split_files`).

    Every frame is held out exactly once, so out-of-fold predictions cover the whole training set.
    """
    files = sorted(files)
    random.Random(seed).shuffle(files)
    return [files[k::n_folds] for k in range(n_folds)]


def channels(image: np.ndarray) -> np.ndarray:
    """float32 (2, H, W): the raw frame and its limb-darkening-corrected version, both scaled to [0, 1]."""
    return np.stack([image, flatten(image)]).astype(np.float32) / 255.0


def soft_target(annotations: Annotations, file_name: str) -> np.ndarray:
    """float32 (H, W) in [0, 1]: the fraction of annotators who marked each pixel as filament.

    Up to three annotators labelled the same frame independently; averaging them keeps the
    disagreement as a soft label instead of arbitrarily trusting one of them.
    """
    per_annotator = annotations.rles_for_file(file_name)
    if not per_annotator:
        raise KeyError(f"no annotations for {file_name}")
    masks = [mask_util.decode(rles).any(axis=2) if rles else None for rles in per_annotator]
    shape = next(m.shape for m in masks if m is not None)
    stacked = np.stack([m if m is not None else np.zeros(shape, bool) for m in masks])
    return stacked.mean(axis=0, dtype=np.float32)


class FilamentFrames(Dataset):
    """One item = one physical frame.

    Training mode returns `tiles_per_frame` random tiles from it: (K, 2, tile, tile) inputs and
    (K, tile, tile) targets. Half of the tiles are centred on a filament pixel, because filaments
    cover 0.4 % of the disk and purely random tiles would mostly be empty. Evaluation mode returns
    the whole frame, (2, H, W) and (H, W).
    """

    def __init__(
        self,
        files: list[str],
        image_dir: Path,
        annotations: Annotations,
        tile: int = 512,
        tiles_per_frame: int = 4,
        train: bool = True,
    ):
        self.files, self.image_dir, self.annotations = files, Path(image_dir), annotations
        self.tile, self.tiles_per_frame, self.train = tile, tiles_per_frame, train

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, index: int):
        x = channels(read_gray(self.image_dir / self.files[index]))
        y = soft_target(self.annotations, self.files[index])
        if not self.train:
            return torch.from_numpy(x), torch.from_numpy(y)
        # torch seeds each DataLoader worker differently every epoch, so tiles differ across epochs
        rng = np.random.default_rng(int(torch.randint(0, 2**31, (1,))))
        tiles = [self.sample_tile(x, y, rng) for _ in range(self.tiles_per_frame)]
        return torch.from_numpy(np.stack([t[0] for t in tiles])), torch.from_numpy(np.stack([t[1] for t in tiles]))

    def sample_tile(self, x: np.ndarray, y: np.ndarray, rng: np.random.Generator):
        positives = np.flatnonzero(y.ravel() > 0.5)
        if len(positives) and rng.random() < 0.5:
            centre = positives[rng.integers(len(positives))]
        else:
            on_disk = np.flatnonzero(x[1].ravel() < OFF_DISK_VALUE)
            centre = on_disk[rng.integers(len(on_disk))] if len(on_disk) else rng.integers(y.size)
        height, width = y.shape
        row, col = divmod(int(centre), width)
        top = int(np.clip(row - self.tile // 2, 0, height - self.tile))
        left = int(np.clip(col - self.tile // 2, 0, width - self.tile))
        tile_x = x[:, top : top + self.tile, left : left + self.tile]
        tile_y = y[top : top + self.tile, left : left + self.tile]
        # only masks are scored, so every rotation and flip is a valid augmentation
        k = int(rng.integers(4))
        tile_x, tile_y = np.rot90(tile_x, k, axes=(1, 2)), np.rot90(tile_y, k)
        if rng.random() < 0.5:
            tile_x, tile_y = tile_x[:, :, ::-1], tile_y[:, ::-1]
        return np.ascontiguousarray(tile_x), np.ascontiguousarray(tile_y)


def worker_init(_: int) -> None:
    """OpenCV's own thread pool and forked DataLoader workers do not mix; one thread per worker is plenty."""
    cv2.setNumThreads(0)


def collate_tiles(batch):
    """Merge the per-frame tile stacks into one batch: (B*K, 2, tile, tile), (B*K, tile, tile)."""
    xs, ys = zip(*batch)
    return torch.cat(xs), torch.cat(ys)

In [ ]:
%%writefile cnn_baseline/filtering.py
"""A second stage that decides, per predicted component, whether it is a filament.

The minimum-area cut is a crude false-positive filter: at 600 px it removes a fifth of the real
filaments to get rid of hallucinations. A small classifier on the component's shape, contrast,
position and the model's confidence separates them better and lets the area cut drop.

It is trained on the validation frames, the only frames whose predictions are honest, and its
cutoff is chosen on out-of-fold scores so the validation PQ it reports is not fitted to itself.
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold

from error_analysis.instances import FrameContext, features, overlaps
from preprocess.disk import disk_geometry, flatten
from scoring.metric import IOU_THRESHOLD, InstanceCrop

FEATURES = ["log_area", "elongation", "darkness", "limb", "mean_prob", "max_prob"]
MIN_TRAINING_COMPONENTS = 60  # below this (or with one class only) the filter is not fitted


def frame_context(name: str, image: np.ndarray, prob: np.ndarray) -> FrameContext:
    cx, cy, radius = disk_geometry(image)
    return FrameContext(name=name, flat=flatten(image), centre=(cx, cy), radius=radius, prob=prob)


def component_table(ctx: FrameContext, labels: np.ndarray, boxes: np.ndarray) -> pd.DataFrame:
    """One row per predicted component with the FEATURES columns (and frame / component id)."""
    rows = []
    for k, (x, y, w, h) in enumerate(boxes, start=1):
        crop = labels[y : y + h, x : x + w] == k
        rows.append({"frame": ctx.name, "component": k, **features(crop, int(x), int(y), ctx)})
    table = pd.DataFrame(
        rows,
        columns=["frame", "component", "area", "cx", "cy", "elongation", "darkness", "limb", "mean_prob", "max_prob"],
    )
    table["log_area"] = np.log(table["area"].astype(float).clip(lower=1))
    return table


def matched_by_any_annotator(
    gt_crops_per_annotator: list[list[InstanceCrop]], labels: np.ndarray, areas: np.ndarray
) -> np.ndarray:
    """bool per component: IoU > 0.5 with a filament of at least one annotator.

    Such a component raises PQ whenever it is kept (its IoU outweighs the annotators it misses); a
    component matching nobody can only lower it. That makes this the right target for the filter.
    """
    keep = np.zeros(len(areas), dtype=bool)
    for crops in gt_crops_per_annotator:
        if crops and len(areas):
            _, iou = overlaps(crops, labels, areas)
            keep |= (iou > IOU_THRESHOLD).any(axis=0)
    return keep


class ComponentFilter:
    """Gradient boosting on `features`; `cutoff` is the probability below which a component is dropped."""

    def __init__(self, cutoff: float = 0.5, features: list[str] = FEATURES):
        self.cutoff, self.features = cutoff, list(features)
        self.model = GradientBoostingClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=0)

    def fit(self, table: pd.DataFrame, target: np.ndarray) -> "ComponentFilter":
        self.model.fit(table[self.features].to_numpy(), np.asarray(target, dtype=int))
        return self

    def scores(self, table: pd.DataFrame) -> np.ndarray:
        if not len(table):
            return np.zeros(0)
        return self.model.predict_proba(table[self.features].to_numpy())[:, 1]

    def keep(self, table: pd.DataFrame) -> np.ndarray:
        return self.scores(table) >= self.cutoff


def out_of_fold_scores(
    table: pd.DataFrame, target: np.ndarray, n_splits: int = 5, features: list[str] = FEATURES
) -> np.ndarray:
    """Filter probabilities for every component, each predicted by a model that never saw its frame."""
    scores = np.zeros(len(table))
    groups = table["frame"].to_numpy()
    splits = min(n_splits, len(np.unique(groups)))
    for train_idx, test_idx in GroupKFold(n_splits=splits).split(table, target, groups):
        model = ComponentFilter(features=features).fit(table.iloc[train_idx], target[train_idx])
        scores[test_idx] = model.scores(table.iloc[test_idx])
    return scores


def trainable(target: np.ndarray) -> bool:
    return len(target) >= MIN_TRAINING_COMPONENTS and 0 < target.sum() < len(target)

In [ ]:
%%writefile cnn_baseline/pairs.py
"""Which kept components belong to the same filament: features of nearby pairs, and joining by a decision.

Joining kept components within a fixed distance helped (`variants.join`), but distance alone cannot tell
how two pieces relate. Physically a filament breaks in two ways:

- **along its spine**, where the material thins out: the pieces line up end to end, and the gap runs
  along their common axis, often still a little darker than the surroundings;
- **at a barb**, a short side extension from the spine down to the chromosphere: a small piece next to
  the *side* of the spine, at an angle to it.

Two separate filaments, in contrast, lie side by side or end to end in different filament channels. The
features below describe a pair in those terms: distance, the angles between the axes and the gap, whether
the gap leaves the larger piece at its end, how dark and how confident the gap is, and the direction of
the chromospheric fibrils around the gap (fibrils in a filament channel run along the channel).
"""

import cv2
import numpy as np
import pandas as pd
from scipy import ndimage

from cnn_baseline.postprocess import select
from cnn_baseline.variants import label_stats, merge
from error_analysis.instances import FrameContext, overlaps
from preprocess.disk import REL_SCALE
from scoring.metric import InstanceCrop

PAIR_FEATURES = [
    "dist",
    "angle_diff",
    "gap_large",
    "gap_small",
    "end_large",
    "end_small",
    "path_dark",
    "path_prob_mean",
    "path_prob_min",
    "fibril_angle",
    "fibril_coherence",
    "log_area_large",
    "log_area_small",
    "elong_large",
    "elong_small",
    "limb",
]
ORIENTED = 2.0  # elongation above which a piece has a meaningful axis
FIBRIL_HALF = 20  # half-width (px) of the window around the gap in which the fibril direction is measured


def line_angle(a, b):
    """Angle between two undirected lines given by their angles (radians), in [0, pi / 2]."""
    d = np.abs(np.asarray(a) - np.asarray(b)) % np.pi
    return np.minimum(d, np.pi - d)


def component_axes(labels: np.ndarray, n: int) -> pd.DataFrame:
    """Per instance 1..n: centroid, principal-axis direction (unit vector, angle in [0, pi)), elongation and
    the extent along the axis (lo..hi, relative to the centroid)."""
    _, boxes = label_stats(labels, n)
    rows = []
    for k, (x, y, w, h) in enumerate(boxes, start=1):
        ys, xs = np.nonzero(labels[y : y + h, x : x + w] == k)
        xs, ys = (xs + x).astype(float), (ys + y).astype(float)
        cx, cy = xs.mean(), ys.mean()
        if len(xs) > 2:
            values, vectors = np.linalg.eigh(np.cov(np.stack([xs, ys])))
            ux, uy = vectors[:, 1]
            elongation = float(np.sqrt(max(values[1], 1e-9) / max(values[0], 1e-9)))
        else:
            ux, uy, elongation = 1.0, 0.0, 1.0
        proj = (xs - cx) * ux + (ys - cy) * uy
        rows.append(
            {
                "cx": cx,
                "cy": cy,
                "ux": ux,
                "uy": uy,
                "angle": float(np.arctan2(uy, ux) % np.pi),
                "elongation": elongation,
                "lo": proj.min(),
                "hi": proj.max(),
                "area": len(xs),
            }
        )
    return pd.DataFrame(rows, columns=["cx", "cy", "ux", "uy", "angle", "elongation", "lo", "hi", "area"])


def pairs(labels: np.ndarray, radius: int) -> pd.DataFrame:
    """Every pair a < b of instances within `radius` px: their distance and nearest pixels (ax, ay), (bx, by)."""
    n = int(labels.max())
    _, boxes = label_stats(labels, n)
    rows = []
    for a, (x, y, w, h) in enumerate(boxes, start=1):
        x0, y0 = max(x - radius, 0), max(y - radius, 0)
        x1, y1 = min(x + w + radius, labels.shape[1]), min(y + h + radius, labels.shape[0])
        window = labels[y0:y1, x0:x1]
        dist, (iy, ix) = ndimage.distance_transform_edt(window != a, return_indices=True)
        for b in np.unique(window[(dist <= radius) & (window > a)]):
            d = np.where(window == b, dist, np.inf)
            by, bx = np.unravel_index(int(np.argmin(d)), d.shape)
            rows.append(
                {
                    "a": a,
                    "b": int(b),
                    "dist": float(d[by, bx]),
                    "ax": int(ix[by, bx]) + x0,
                    "ay": int(iy[by, bx]) + y0,
                    "bx": int(bx) + x0,
                    "by": int(by) + y0,
                }
            )
    return pd.DataFrame(rows, columns=["a", "b", "dist", "ax", "ay", "bx", "by"])


def path_values(image: np.ndarray, p: tuple[int, int], q: tuple[int, int]) -> np.ndarray:
    """Values of `image` on the straight path from pixel p to pixel q (x, y), endpoints excluded when possible."""
    m = max(int(np.hypot(q[0] - p[0], q[1] - p[1])) - 1, 1)
    t = np.linspace(0, 1, m + 2)[1:-1]
    xs = np.rint(p[0] + t * (q[0] - p[0])).astype(int)
    ys = np.rint(p[1] + t * (q[1] - p[1])).astype(int)
    return image[ys, xs]


def fibril_direction(image: np.ndarray, x: int, y: int, exclude: np.ndarray, half: int = FIBRIL_HALF):
    """(angle of the dominant line direction in [0, pi), coherence in [0, 1]) around (x, y).

    Structure tensor of the image gradients: the dominant gradient direction is across the fibrils, so the
    fibrils run perpendicular to it. Pixels where `exclude` is set (the components themselves, whose edges
    would otherwise dominate) do not count. Coherence 0 = no preferred direction, 1 = perfectly parallel lines.
    """
    x0, y0 = max(x - half, 0), max(y - half, 0)
    x1, y1 = min(x + half + 1, image.shape[1]), min(y + half + 1, image.shape[0])
    window = image[y0:y1, x0:x1].astype(np.float32)
    gx = cv2.Sobel(window, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(window, cv2.CV_32F, 0, 1, ksize=3)
    weight = (~exclude[y0:y1, x0:x1]).astype(np.float32)
    jxx, jyy, jxy = (float((weight * g).sum()) for g in (gx * gx, gy * gy, gx * gy))
    if jxx + jyy <= 0:
        return 0.0, 0.0
    gradient = 0.5 * np.arctan2(2 * jxy, jxx - jyy)
    coherence = np.hypot(jxx - jyy, 2 * jxy) / (jxx + jyy)
    return float((gradient + np.pi / 2) % np.pi), float(coherence)


def end_position(axes: pd.Series, x: float, y: float) -> float:
    """Where (x, y) projects along a piece's axis: 0 at either end, 0.5 in the middle."""
    span = axes["hi"] - axes["lo"]
    if span <= 0:
        return 0.5
    t = ((x - axes["cx"]) * axes["ux"] + (y - axes["cy"]) * axes["uy"] - axes["lo"]) / span
    t = float(np.clip(t, 0, 1))
    return min(t, 1 - t)


def pair_table(ctx: FrameContext, labels: np.ndarray, radius: int = 50) -> pd.DataFrame:
    """One row per pair of instances within `radius` px: PAIR_FEATURES plus frame, a, b."""
    near = pairs(labels, radius)
    if not len(near):
        return pd.DataFrame(columns=["frame", "a", "b", *PAIR_FEATURES])
    axes = component_axes(labels, int(labels.max()))
    exclude = cv2.dilate((labels > 0).astype(np.uint8), np.ones((7, 7), np.uint8)) > 0
    rows = []
    for p in near.itertuples(index=False):
        A, B = axes.iloc[p.a - 1], axes.iloc[p.b - 1]
        a_large = A["area"] >= B["area"]
        L, S = (A, B) if a_large else (B, A)
        pl, ps = ((p.ax, p.ay), (p.bx, p.by)) if a_large else ((p.bx, p.by), (p.ax, p.ay))
        gap = float(np.arctan2(p.by - p.ay, p.bx - p.ax) % np.pi)
        mid_x, mid_y = (p.ax + p.bx) // 2, (p.ay + p.by) // 2
        flat = path_values(ctx.flat, pl, ps).astype(float)
        prob = path_values(ctx.prob, pl, ps).astype(float) / 255
        fibril, coherence = fibril_direction(ctx.flat, mid_x, mid_y, exclude)
        rows.append(
            {
                "frame": ctx.name,
                "a": p.a,
                "b": p.b,
                "dist": p.dist,
                "angle_diff": float(line_angle(L["angle"], S["angle"])),
                "gap_large": float(line_angle(gap, L["angle"])),
                "gap_small": float(line_angle(gap, S["angle"])),
                "end_large": end_position(L, *pl),
                "end_small": end_position(S, *ps),
                "path_dark": float(1 - flat.mean() / REL_SCALE),
                "path_prob_mean": float(prob.mean()),
                "path_prob_min": float(prob.min()),
                "fibril_angle": float(line_angle(fibril, gap)),
                "fibril_coherence": coherence,
                "log_area_large": float(np.log(L["area"])),
                "log_area_small": float(np.log(S["area"])),
                "elong_large": L["elongation"],
                "elong_small": S["elongation"],
                "limb": float(np.hypot(mid_x - ctx.centre[0], mid_y - ctx.centre[1]) / ctx.radius)
                if ctx.radius
                else np.nan,
            }
        )
    return pd.DataFrame(rows, columns=["frame", "a", "b", *PAIR_FEATURES])


def same_filament(
    gt_crops_per_annotator: list[list[InstanceCrop]],
    labels: np.ndarray,
    areas: np.ndarray,
    table: pd.DataFrame,
    share: float = 0.3,
) -> tuple[np.ndarray, np.ndarray]:
    """(same, different) per pair: for some annotator, `share` or more of both pieces lies in one filament
    (same), or in two different filaments of theirs (different, and never same)."""
    same = np.zeros(len(table), bool)
    different = np.zeros(len(table), bool)
    if not len(table):
        return same, different
    a, b = table["a"].to_numpy(int) - 1, table["b"].to_numpy(int) - 1
    for crops in gt_crops_per_annotator:
        if not crops:
            continue
        intersection, _ = overlaps(crops, labels, areas)
        inside = intersection / np.maximum(areas, 1)[None, :] >= share  # (n_gt, n_pred)
        same |= (inside[:, a] & inside[:, b]).any(axis=0)
        different |= inside[:, a].any(axis=0) & inside[:, b].any(axis=0)
    return same, different & ~same


def join_pairs(labels: np.ndarray, table: pd.DataFrame, decide: np.ndarray):
    """Merge the pairs of `table` where `decide` is True (transitively); renumber largest first."""
    chosen = table.loc[np.asarray(decide, bool), ["a", "b"]].to_numpy(int)
    return merge(labels, [tuple(link) for link in chosen])


def aligned_end_to_end(table: pd.DataFrame, max_angle: float = np.radians(30), max_end: float = 0.15) -> np.ndarray:
    """Pairs that continue each other along the spine: the gap leaves the larger piece at its end along its
    axis, and also follows the smaller piece's axis when that piece is elongated enough to have one."""
    large = (table["elong_large"] >= ORIENTED) & (table["gap_large"] <= max_angle) & (table["end_large"] <= max_end)
    small = (table["elong_small"] < ORIENTED) | (table["gap_small"] <= max_angle)
    return (large & small).to_numpy()


def join_rule(table: pd.DataFrame, near: int = 10, far: int = 50) -> np.ndarray:
    """The rule of `join_lab.py`: pieces that continue each other end to end along the axis join up to `far`
    px apart, any two pieces up to `near` px (barbs, blobs)."""
    return ((table["dist"] <= near) | ((table["dist"] <= far) & aligned_end_to_end(table))).to_numpy()


def rescue(ctx: FrameContext, labels: np.ndarray, keep: np.ndarray, radius: int = 50):
    """Kept instances joined by `join_rule`, with each other and with dropped instances that continue them.

    `labels` holds every candidate 1..K before the filter and `keep` (bool[K]) the filter's decision. A dropped
    piece comes back only as part of a group that contains a kept one; groups of dropped pieces stay dropped.
    """
    keep = np.asarray(keep, bool)
    assert int(labels.max()) == len(keep), "one decision per candidate"
    table = pair_table(ctx, labels, radius)
    if len(table):
        a, b = table["a"].to_numpy(int) - 1, table["b"].to_numpy(int) - 1
        joined, areas, boxes = join_pairs(labels, table, join_rule(table) & (keep[a] | keep[b]))
    else:
        joined, areas, boxes = merge(labels, [])
    has_kept = np.zeros(len(areas) + 1, bool)
    has_kept[np.unique(joined[np.r_[False, keep][labels]])] = True
    has_kept[0] = False
    return select(joined, areas, boxes, has_kept[1:])

In [ ]:
%%writefile cnn_baseline/postprocess.py
"""Probability map -> filament instances: (hysteresis) threshold and connected components."""

import cv2
import numpy as np

from preprocess.disk import disk_mask

MAX_INSTANCES = 40  # ground truth never exceeds 26 filaments per frame
DISK_SLACK = 1.02  # keep a sliver past the estimated limb so filaments touching it are not clipped


def on_disk(prob: np.ndarray, image: np.ndarray) -> np.ndarray:
    """`prob` with everything off the solar disk set to 0.

    Training tiles are centred on the disk, so the network barely sees the sky beyond the limb and its
    output there hovers around 0.5 -- enough to pass a 0.5 threshold as large spurious blobs.
    """
    return prob * disk_mask(image, shrink=DISK_SLACK)


def instances(prob: np.ndarray, threshold: float, min_area: int, low: float | None = None):
    """(label map with 0 = background and 1..K, areas[K], boxes[K, 4] as x, y, w, h), largest first.

    `prob` is uint8 (H, W) holding probability * 255, as `training.predict` yields it.

    With `low` (< threshold) the threshold is hysteretic: a component of `prob >= low` is kept whole
    when any of its pixels reaches `threshold`. Filaments fade towards their ends, so a single
    threshold keeps the confident core and drops the faint tails; hysteresis follows them.
    """
    high = prob >= round(threshold * 255)
    if low is not None and low < threshold:
        n_low, low_labels = cv2.connectedComponents((prob >= round(low * 255)).astype(np.uint8), connectivity=8)
        seeded = np.zeros(n_low, dtype=bool)
        seeded[np.unique(low_labels[high])] = True
        seeded[0] = False
        foreground = seeded[low_labels].astype(np.uint8)
    else:
        foreground = high.astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(foreground, connectivity=8)
    areas = stats[:, cv2.CC_STAT_AREA]
    keep = np.flatnonzero(areas >= min_area)
    keep = keep[keep != 0]  # component 0 is the background
    keep = keep[np.argsort(areas[keep])[::-1][:MAX_INSTANCES]]
    relabel = np.zeros(n, dtype=np.int32)
    relabel[keep] = np.arange(1, len(keep) + 1)
    return relabel[labels], areas[keep], stats[keep, :4]


def select(labels: np.ndarray, areas: np.ndarray, boxes: np.ndarray, keep: np.ndarray):
    """Drop the instances where `keep` (bool per instance 1..K) is False; renumber the rest 1..K'."""
    keep = np.asarray(keep, dtype=bool)
    relabel = np.zeros(len(areas) + 1, dtype=np.int32)
    relabel[1:][keep] = np.arange(1, int(keep.sum()) + 1)
    return relabel[labels], areas[keep], boxes[keep]

In [ ]:
%%writefile cnn_baseline/training.py
"""Training loop and full-frame inference for a binary segmentation network."""

import os
import time
from collections.abc import Iterable, Iterator

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader


def pick_device() -> torch.device:
    """CUDA if there is one; locally fall back to Apple's MPS or the CPU.

    On Kaggle a missing GPU is an error, not a fallback: an account without phone verification (or
    without GPU quota) gets a CPU machine for a GPU notebook without any warning, and this U-Net then
    needs ~3.5 h per epoch instead of minutes.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        raise RuntimeError(
            "Kaggle gave this notebook no GPU. Check that the account is phone-verified and has GPU quota "
            "left (Settings on kaggle.com); stopping instead of training on the CPU."
        )
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def bce_dice_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """BCE for per-pixel accuracy plus soft Dice for the 0.4 % of pixels that are filament.

    Both accept soft targets, so annotator disagreement flows into the loss as fractional labels.
    """
    logits, target = logits.squeeze(1).float(), target.float()
    bce = F.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    intersection = (prob * target).sum(dim=(1, 2))
    dice = 1 - (2 * intersection + 1) / (prob.sum(dim=(1, 2)) + target.sum(dim=(1, 2)) + 1)
    return bce + dice.mean()


def fit(
    model: nn.Module,
    loader: DataLoader,
    epochs: int,
    lr: float = 1e-3,
    device: torch.device | None = None,
    log=print,
) -> list[float]:
    """AdamW + one-cycle schedule, mixed precision on CUDA. Returns the mean loss of each epoch."""
    device = device or pick_device()
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * len(loader))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        start, total, steps = time.time(), 0.0, 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = bce_dice_loss(model(x), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            schedule.step()
            total, steps = total + loss.item(), steps + 1
        history.append(total / max(steps, 1))
        log(
            f"epoch {epoch}/{epochs}  loss {history[-1]:.4f}  lr {schedule.get_last_lr()[0]:.2e}  {time.time() - start:.0f}s"
        )
    return history


@torch.no_grad()
def predict(
    model: nn.Module, frames: Iterable[torch.Tensor], device: torch.device | None = None
) -> Iterator[np.ndarray]:
    """Yield a uint8 (H, W) probability map (probability * 255) for each (2, H, W) frame."""
    device = device or pick_device()
    model.to(device).eval()
    use_amp = device.type == "cuda"
    for frame in frames:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(frame.unsqueeze(0).to(device))
        prob = torch.sigmoid(logits.float()).squeeze().mul(255).round().to(torch.uint8)
        yield prob.cpu().numpy()


D4 = [(k, flip) for flip in (False, True) for k in range(4)]  # (quarter turns, mirrored); identity first


@torch.no_grad()
def predict_tta(
    model: nn.Module, frames: Iterable[torch.Tensor], device: torch.device | None = None, views: int = 8
) -> Iterator[tuple[np.ndarray, np.ndarray]]:
    """Yield (plain, averaged) uint8 maps per (2, H, W) frame: the identity view alone, as `predict` gives it,
    and the mean probability over the first `views` rotations / mirror images, each turned back first.

    A filament has no preferred orientation in the image, so the network should answer the same for every
    view; averaging removes the part of its output that depends on the view.
    """
    device = device or pick_device()
    model.to(device).eval()
    use_amp = device.type == "cuda"
    for frame in frames:
        plain, total = None, None
        for k, flip in D4[:views]:
            x = torch.rot90(frame.flip(-1) if flip else frame, k, dims=(-2, -1))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x.unsqueeze(0).to(device))
            prob = torch.rot90(torch.sigmoid(logits.float()).squeeze(), -k, dims=(-2, -1))
            prob = prob.flip(-1) if flip else prob
            plain = prob if plain is None else plain
            total = prob.clone() if total is None else total + prob
        yield (
            plain.mul(255).round().to(torch.uint8).cpu().numpy(),
            total.div(views).mul(255).round().to(torch.uint8).cpu().numpy(),
        )

In [ ]:
%%writefile cnn_baseline/variants.py
"""Post-processing variants compared against A0 (`predict.py`) on the saved probability maps.

A variant turns a frame's probability map into instances (labels 1..K, areas, boxes). Three kinds:

- **keep rule**: which components survive. A0 keeps those whose filter probability of "matched by any
  annotator" passes a cutoff tuned on validation. `expected_gain` gives the target that PQ actually
  rewards, so the cutoff follows from the metric instead of a sweep.
- **grow**: kept components extend into neighbouring pixels where the network is still fairly confident
  (near misses: the core is found, the faint ends are not).
- **join**: kept components close to each other become one instance (fragments: annotators draw a filament
  across the gaps where its material thins out).
"""

import cv2
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GroupKFold

from cnn_baseline.filtering import FEATURES
from cnn_baseline.postprocess import instances, select
from error_analysis.instances import overlaps
from scoring.metric import IOU_THRESHOLD, InstanceCrop, PanopticQuality


def label_stats(labels: np.ndarray, n: int) -> tuple[np.ndarray, np.ndarray]:
    """(areas[n], boxes[n, 4] as x, y, w, h) of the instances 1..n of a label map (empty ones: area 0)."""
    ys, xs = np.nonzero(labels)
    ids = labels[ys, xs] - 1
    areas = np.bincount(ids, minlength=n)[:n]
    x0, y0 = np.full(n, labels.shape[1]), np.full(n, labels.shape[0])
    x1, y1 = np.full(n, -1), np.full(n, -1)
    np.minimum.at(x0, ids, xs)
    np.minimum.at(y0, ids, ys)
    np.maximum.at(x1, ids, xs)
    np.maximum.at(y1, ids, ys)
    boxes = np.stack([x0, y0, x1 - x0 + 1, y1 - y0 + 1], axis=1)
    boxes[areas == 0] = 0
    return areas, boxes


def renumber(labels: np.ndarray, mapping: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Apply `mapping` (old label -> group, 0 stays 0), then number the groups 1..K largest first."""
    grouped = mapping[labels]
    groups = np.unique(mapping[1:])
    groups = groups[groups > 0]
    compact = np.zeros(int(mapping.max()) + 1, dtype=np.int32)
    compact[groups] = np.arange(1, len(groups) + 1)
    grouped = compact[grouped]
    areas, _ = label_stats(grouped, len(groups))
    order = np.argsort(areas, kind="stable")[::-1]
    by_size = np.zeros(len(groups) + 1, dtype=np.int32)
    by_size[order + 1] = np.arange(1, len(groups) + 1)
    out = by_size[grouped]
    areas, boxes = label_stats(out, len(groups))
    return out, areas, boxes


def grow(labels: np.ndarray, prob: np.ndarray, low: float, steps: int):
    """Extend every instance, one pixel ring per step, into unlabelled pixels with prob >= low.

    Instances never merge: a pixel reached by two instances in the same step goes to the higher label.
    """
    allowed = prob >= round(low * 255)
    grown = labels.astype(np.uint16)
    kernel = np.ones((3, 3), np.uint8)
    for _ in range(steps):
        reach = cv2.dilate(grown, kernel)
        new = (grown == 0) & allowed & (reach > 0)
        if not new.any():
            break
        grown[new] = reach[new]
    n = int(labels.max())
    areas, boxes = label_stats(grown.astype(np.int32), n)
    return grown.astype(np.int32), areas, boxes


def merge(labels: np.ndarray, links) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Merge the instances of each (a, b) in `links`, transitively; renumber largest first. No pixels are added."""
    n = int(labels.max())
    parent = np.arange(n + 1)

    def root(k):
        while parent[k] != k:
            parent[k] = parent[parent[k]]
            k = parent[k]
        return k

    for a, b in links:
        parent[root(int(b))] = root(int(a))
    return renumber(labels, np.array([root(k) for k in range(n + 1)]))


def join(labels: np.ndarray, radius: int):
    """Merge instances whose pixels come within `radius` px of each other (transitively); no pixels are added."""
    _, boxes = label_stats(labels, int(labels.max()))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    links = []
    for k, (x, y, w, h) in enumerate(boxes, start=1):
        x0, y0 = max(x - radius, 0), max(y - radius, 0)
        x1, y1 = min(x + w + radius, labels.shape[1]), min(y + h + radius, labels.shape[0])
        window = labels[y0:y1, x0:x1]
        near = cv2.dilate((window == k).astype(np.uint8), kernel) > 0
        links += [(k, other) for other in np.unique(window[near]) if other and other != k]
    return merge(labels, links)


def expected_gain(gt_crops_per_annotator: list[list[InstanceCrop]], labels: np.ndarray, areas: np.ndarray):
    """Per component: mean over the frame's annotators of IoU if it matches their filament (IoU > 0.5), else 0.

    PQ = sum(IoU) / (TP + FP/2 + FN/2). For each annotator a kept component either matches (IoU joins the
    numerator, an FN becomes a TP) or not (one more FP); either way the denominator grows by 1/2. Keeping
    it therefore raises PQ exactly when its mean gain per annotator exceeds PQ / 2.
    """
    gain = np.zeros(len(areas))
    if not len(areas) or not gt_crops_per_annotator:
        return gain
    for crops in gt_crops_per_annotator:
        if crops:
            _, iou = overlaps(crops, labels, areas)
            best = iou.max(axis=0)
            gain += np.where(best > IOU_THRESHOLD, best, 0.0)
    return gain / len(gt_crops_per_annotator)


class GainRegressor:
    """Gradient boosting that predicts `expected_gain` from the component features."""

    def __init__(self, features: list[str] = FEATURES):
        self.features = list(features)
        self.model = GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=0)

    def fit(self, table: pd.DataFrame, gain: np.ndarray) -> "GainRegressor":
        self.model.fit(table[self.features].to_numpy(), gain)
        return self

    def predict(self, table: pd.DataFrame) -> np.ndarray:
        return self.model.predict(table[self.features].to_numpy()) if len(table) else np.zeros(0)


def out_of_fold_gain(table: pd.DataFrame, gain: np.ndarray, n_splits: int = 5) -> np.ndarray:
    """Predicted gain for every component from a model that never saw its frame (same folds as the filter)."""
    pred = np.zeros(len(table))
    groups = table["frame"].to_numpy()
    splits = min(n_splits, len(np.unique(groups)))
    for train_idx, test_idx in GroupKFold(n_splits=splits).split(table, gain, groups):
        pred[test_idx] = GainRegressor().fit(table.iloc[train_idx], gain[train_idx]).predict(table.iloc[test_idx])
    return pred


def score_frames(predictions: dict, truth: dict, files: list[str]) -> pd.DataFrame:
    """Per-frame PQ terms (as `importance.frame_terms`) of ready-made predictions {frame: (labels, areas, boxes)}."""
    rows = []
    for f in files:
        labels, areas, _ = predictions[f]
        pq = PanopticQuality()
        for gt_crops in truth[f]:
            pq.update(gt_crops, labels, areas)
        rows.append({"frame": f, "tp": pq.tp, "fp": pq.fp, "fn": pq.fn, "iou_sum": pq.iou_sum})
    return pd.DataFrame(rows, columns=["frame", "tp", "fp", "fn", "iou_sum"])


def strict_keep(prob: np.ndarray, low: float = 0.35, peak: float = 0.99, mean: float = 0.85, min_area: int = 200):
    """The keep rule of the public "OOF 0.45 vs Public 0.39" notebook: components of p >= `low` (8-connected)
    whose peak reaches `peak`, whose mean probability is at least `mean` and whose area is at least `min_area`."""
    labels, areas, boxes = instances(prob, threshold=peak, min_area=min_area, low=low)
    sums = np.bincount(labels.ravel(), weights=prob.ravel().astype(float), minlength=len(areas) + 1)[1:]
    return select(labels, areas, boxes, sums / np.maximum(areas, 1) / 255 >= mean)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from cnn_baseline.data import channels, split_files
from cnn_baseline.filtering import component_table, frame_context
from cnn_baseline.pairs import join_pairs, join_rule, pair_table
from cnn_baseline.postprocess import instances, on_disk
from cnn_baseline.training import predict
from dataset.annotations import load_annotations
from dataset.paths import find_data_root, find_input, output_dir
from models.unet import UNet
from preprocess.disk import disk_geometry
from preprocess.images import read_gray
from scoring import submission
from scoring.metric import PanopticQuality, instance_crops

SEED, VALIDATION_FRACTION = 0, 0.15  # the split the network was trained with: these frames were not seen
A0 = {"threshold": 0.5, "min_area": 300, "low": 0.3}
MAX_TRIES = 20  # validation frames tried when looking for one where joining acts


def show(name: str) -> None:
    """Save the current figure next to the notebook's outputs, then display it."""
    plt.savefig(output_dir() / f"pipeline_{name}.png", dpi=100)
    plt.show()


root = find_data_root()
train_dir = root / "train" / "train_images"
annotations = load_annotations(next((root / "train").glob("*.json")))
_, val_files = split_files(sorted(p.name for p in train_dir.iterdir()), VALIDATION_FRACTION, SEED)
model = UNet(in_channels=2, base=32, depth=4)
model.load_state_dict(torch.load(find_input("unet.pt"), map_location="cpu"))


def run_frame(name: str):
    """image, network probability map, frame context, A0 instances and joined instances of one frame."""
    image = read_gray(train_dir / name)
    prob = on_disk(next(predict(model, [torch.from_numpy(channels(image))], torch.device("cpu"))), image)
    a0 = instances(prob, **A0)
    ctx = frame_context(name, image, prob)
    pairs = pair_table(ctx, a0[0], 50)
    joined = join_pairs(a0[0], pairs, join_rule(pairs)) if len(pairs) else a0
    return image, prob, ctx, a0, joined

**Which frame.** Chosen by a fixed rule, for illustration: among the validation frames with the most
annotators (three on the real data), the first one where joining merges at least two pieces. If none of the
first frames tried does, the first one is shown. It is not a typical frame; the report gives 707-frame scores.

In [ ]:
by_annotators = sorted(val_files, key=lambda f: -len(annotations.rles_for_file(f)))  # stable: keeps name order
for frame in by_annotators[:MAX_TRIES]:
    image, prob, ctx, a0, joined = run_frame(frame)
    if len(joined[1]) < len(a0[1]):
        break
else:
    frame = by_annotators[0]
    image, prob, ctx, a0, joined = run_frame(frame)
print(frame, image.shape, image.dtype, f"({len(annotations.rles_for_file(frame))} annotators)")

## 1. Disk and limb darkening

Limb darkening is multiplicative, so the corrected channel is the frame *divided* by a smooth background
(a normalised convolution over the disk): a filament then reads ~0.88 of its surroundings at the centre
and at the limb alike. The network gets both the raw and the corrected frame.

In [ ]:
cx, cy, radius = disk_geometry(image)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, channel, title in zip(axes, channels(image), ["raw frame", "limb-darkening corrected"]):
    ax.imshow(channel, cmap="gray")
    ax.add_patch(plt.Circle((cx, cy), radius, fill=False, color="tab:orange", lw=0.8))
    ax.set_title(title)
    ax.axis("off")
show("1_disk")
print(f"disk centre ({cx:.0f}, {cy:.0f}) px, radius {radius:.0f} px")

## 2. Network: probability map

A 4-level U-Net (2 input channels) trained on 512 px tiles with the fraction of annotators as a soft target.
It ran above on CPU on the full 2048 x 2048 frame; off-disk pixels are zeroed.

In [ ]:
print("probability map", prob.shape, prob.dtype, "(probability x 255)")

## 3. Instances: hysteresis, then joining

A component of p >= 0.3 is kept whole when any pixel reaches 0.5, so faint tails stay attached; then pieces
that continue each other end to end along their axes (<= 50 px) or lie within 10 px are joined, the two
ways a filament physically breaks (material thinning along the spine; barbs).

In [ ]:
print(f"A0: {len(a0[1])} instances; after joining: {len(joined[1])}")
truth = annotations.rles_for_file(frame)
first = np.zeros_like(prob, dtype=np.int32)
for k, (x0, y0, mask) in enumerate(instance_crops(truth[0]), start=1):
    first[y0 : y0 + mask.shape[0], x0 : x0 + mask.shape[1]][mask] = k
panels = [
    (prob, "magma", "network probability"),
    (np.ma.masked_equal(a0[0], 0), "tab20", f"A0 instances ({len(a0[1])})"),
    (np.ma.masked_equal(joined[0], 0), "tab20", f"after joining ({len(joined[1])})"),
    (np.ma.masked_equal(first, 0), "tab20", f"annotator 1 ({len(truth[0])})"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, (data, cmap, title) in zip(axes.flat, panels):
    ax.imshow(data, cmap=cmap, interpolation="nearest")
    ax.set_title(title)
    ax.set_xlim(cx - radius, cx + radius)
    ax.set_ylim(cy + radius, cy - radius)
    ax.axis("off")
show("2_instances")

## 4. Component features (the filter's inputs)

The filter predicts whether a component matches any annotator from these features; it is trained out of fold
on all 707 training frames in `final_join.py`, and its cutoff follows the break-even rule
P(match) x IoU > PQ/2. Here are the features of this frame's joined components.

In [ ]:
table = component_table(ctx, joined[0], joined[2])
print(table[["area", "elongation", "darkness", "limb", "mean_prob", "max_prob"]].round(3).to_string())

## 5. Score, as the hosts do: per annotator

PQ = sum of IoU over matches / (TP + FP/2 + FN/2), a match needing IoU > 0.5, accumulated over the
annotators of the frame. One frame is a noisy number; the report gives the 707-frame OOF scores.

In [ ]:
for name, (labels, areas, _) in {"A0": a0, "A0 + join": joined}.items():
    pq = PanopticQuality()
    for rles in truth:
        pq.update(instance_crops(rles), labels, areas)
    print(name, {k: round(v, 3) if isinstance(v, float) else v for k, v in pq.summary().items()})

## 6. Submission rows

One row per filament: `filament_id` = `<image id>_<n>` and the pycocotools-compressed RLE of its mask.

In [ ]:
rows = submission.rows_from_label_map(frame.rsplit(".", 1)[0], joined[0], joined[2])
print(
    len(rows),
    "rows; first:",
    {k: (v[:60] + "...") if isinstance(v, str) and len(v) > 60 else v for k, v in rows[0].items()},
)